# Experiment 03 - PGD Targeted Attack

## Step 1: Define the target voice command

In [ ]:
target_text = "OPEN THE SETTINGS MENU"

print("Target transcription:", target_text)

Target transcription: OPEN THE SETTINGS MENU


## Step 2: Initialize PyTorch and CUDA

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.15.0+cu132
CUDA available: True
Device: cuda
GPU: NVIDIA GeForce RTX 5080 Laptop GPU


## Step 3: Load Wav2Vec2

In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

model_id = "facebook/wav2vec2-base-960h"

processor = Wav2Vec2Processor.from_pretrained(model_id)

model = Wav2Vec2ForCTC.from_pretrained(model_id)
model = model.to(device)
model.eval()

print("Model:", model_id)
print("Device:", next(model.parameters()).device)
print("Sampling rate:", processor.feature_extractor.sampling_rate)
print("Model loaded successfully.")

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 873.97it/s]
[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | Details
---------------------------+---------+--------
wav2vec2.masked_spec_embed | MISSING |        

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: facebook/wav2vec2-base-960h
Device: cuda:0
Sampling rate: 16000
Model loaded successfully.


## Step 4: Load the Original Audio Recording

In [ ]:
import librosa
import numpy as np
from IPython.display import Audio, display

audio_path = "../data/exp3/original.wav"  # Replace with your audio filename

waveform, sample_rate = librosa.load(
    audio_path,
    sr=16000,
    mono=True
)

print("Sample rate:", sample_rate)
print("Duration:", round(len(waveform) / sample_rate, 2), "seconds")
print("Waveform shape:", waveform.shape)

display(Audio(waveform, rate=sample_rate))

Sample rate: 16000
Duration: 4.1 seconds
Waveform shape: (65525,)


## Step 5: Baseline Transcription

In [ ]:
# Prepare audio for Wav2Vec2
inputs = processor(
    waveform,
    sampling_rate=sample_rate,
    return_tensors="pt"
)

input_values = inputs.input_values.to(device)

# Perform baseline inference
with torch.no_grad():
    logits = model(input_values).logits

# Decode predicted transcription
predicted_ids = torch.argmax(logits, dim=-1)
baseline_text = processor.batch_decode(predicted_ids)[0]

print("Baseline transcription:", baseline_text)
print("Target transcription:  ", target_text)

Baseline transcription: A STITCH IN TIME SAVES NINE BUT ACTION SPEAK LOUDER THAN WORDS
Target transcription:   OPEN THE SETTINGS MENU


## Step 6: Encode Target Transcription

In [ ]:
# Convert target command into CTC token IDs
target_ids = processor.tokenizer(
    target_text,
    return_tensors="pt"
).input_ids.to(device)

# Verify encoding
print("Target command:", target_text)
print("Token IDs:", target_ids.tolist())
print("Number of tokens:", target_ids.shape[1])
print("Decoded target:", processor.tokenizer.decode(target_ids[0]))

Target command: OPEN THE SETTINGS MENU
Token IDs: [[8, 23, 5, 9, 4, 6, 11, 5, 4, 12, 5, 6, 6, 10, 9, 21, 12, 4, 17, 5, 9, 16]]
Number of tokens: 22
Decoded target: OPEN THE SETINGS MENU


## Step 7: Configure Targeted PGD Parameters

In [ ]:
# Targeted PGD configuration
epsilon = 0.01
alpha = 0.0005
num_iterations = 500

# Preserve original waveform
original_audio = input_values.detach().clone()

# Initialize adversarial waveform
adversarial_audio = original_audio.clone()

print(f"Epsilon: {epsilon}")
print(f"Step size: {alpha}")
print(f"Iterations: {num_iterations}")
print(f"Target: {target_text}")
print(f"Audio shape: {tuple(original_audio.shape)}")

Epsilon: 0.01
Step size: 0.0005
Iterations: 500
Target: OPEN THE SETTINGS MENU
Audio shape: (1, 65525)


## Step 8: Execute Targeted PGD

In [ ]:

import torch
import torch.nn.functional as F
import time

# ============================================================
# 1. Experiment Configuration
# ============================================================

epsilon = 0.01
alpha = 0.0005
num_iterations = 500

# ============================================================
# 2. Initialize Raw Audio
# ============================================================

# Reconstruct directly from the raw waveform to avoid
# stale or normalized tensors from previous notebook cells.
original_audio = torch.as_tensor(
    waveform,
    dtype=torch.float32,
    device=device
).clone().unsqueeze(0)

# Validate raw audio amplitude
assert original_audio.min().item() >= -1.0, (
    "Original audio minimum is below -1.0"
)

assert original_audio.max().item() <= 1.0, (
    "Original audio maximum is above 1.0"
)

# Initialize adversarial waveform
adversarial_audio = original_audio.detach().clone()

print("Raw audio minimum:", original_audio.min().item())
print("Raw audio maximum:", original_audio.max().item())

# ============================================================
# 3. Initialize Experiment Tracking
# ============================================================

loss_history = []
best_loss = float("inf")
best_audio = None
best_iteration = None

model.eval()
start_time = time.perf_counter()

# ============================================================
# 4. Audio Normalization
# ============================================================

def normalize_audio(audio):
    mean = audio.mean(dim=-1, keepdim=True)

    variance = audio.var(
        dim=-1,
        keepdim=True,
        unbiased=False
    )

    return (audio - mean) / torch.sqrt(variance + 1e-7)

# ============================================================
# 5. Targeted CTC Loss
# ============================================================

def target_ctc_loss(audio):
    normalized = normalize_audio(audio)

    logits = model(normalized).logits

    log_probs = F.log_softmax(
        logits,
        dim=-1
    ).transpose(0, 1)

    input_lengths = torch.full(
        (log_probs.shape[1],),
        log_probs.shape[0],
        dtype=torch.long,
        device=device
    )

    target_lengths = torch.full(
        (target_ids.shape[0],),
        target_ids.shape[1],
        dtype=torch.long,
        device=device
    )

    return F.ctc_loss(
        log_probs,
        target_ids,
        input_lengths,
        target_lengths,
        blank=model.config.pad_token_id,
        reduction="mean",
        zero_infinity=True
    )

# ============================================================
# 6. Targeted PGD Optimization
# ============================================================

for iteration in range(num_iterations):

    # Enable gradient computation
    adversarial_audio = (
        adversarial_audio.detach().requires_grad_(True)
    )

    # Forward pass and loss calculation
    loss = target_ctc_loss(adversarial_audio)

    current_loss = loss.item()
    loss_history.append(current_loss)

    # Preserve best waveform
    if current_loss < best_loss:
        best_loss = current_loss
        best_audio = adversarial_audio.detach().clone()
        best_iteration = iteration + 1

    # Calculate gradient
    gradient = torch.autograd.grad(
        loss,
        adversarial_audio
    )[0]

    with torch.no_grad():

        # Targeted PGD: minimize target CTC loss
        adversarial_audio = (
            adversarial_audio - alpha * gradient.sign()
        )

        # Project perturbation into epsilon constraint
        perturbation = torch.clamp(
            adversarial_audio - original_audio,
            min=-epsilon,
            max=epsilon
        )

        # Enforce valid waveform amplitude
        adversarial_audio = torch.clamp(
            original_audio + perturbation,
            min=-1.0,
            max=1.0
        )

        # Verify epsilon constraint after every update
        current_max_perturbation = (
            adversarial_audio - original_audio
        ).abs().max().item()

        assert current_max_perturbation <= epsilon + 1e-6, (
            "PGD perturbation constraint violated!"
        )

    # Display optimization progress
    if (iteration + 1) % 50 == 0:
        print(
            f"Iteration {iteration + 1:3d}/{num_iterations} "
            f"| Target CTC loss: {current_loss:.4f}"
        )

# ============================================================
# 7. Evaluate Final Waveform
# ============================================================

with torch.no_grad():
    final_loss = target_ctc_loss(adversarial_audio).item()

# Compare final updated waveform against best result
if final_loss < best_loss:
    best_loss = final_loss
    best_audio = adversarial_audio.detach().clone()
    best_iteration = num_iterations + 1

elapsed_time = time.perf_counter() - start_time

# ============================================================
# 8. Decode Best Adversarial Waveform
# ============================================================

with torch.no_grad():
    normalized_best_audio = normalize_audio(best_audio)

    logits = model(normalized_best_audio).logits

    predicted_ids = torch.argmax(
        logits,
        dim=-1
    )

best_transcription = processor.batch_decode(
    predicted_ids
)[0]

# ============================================================
# 9. Validate Perturbation
# ============================================================

delta = best_audio - original_audio

max_perturbation = delta.abs().max().item()

within_constraint = (
    max_perturbation <= epsilon + 1e-6
)

# ============================================================
# 10. Display Results
# ============================================================

print("\n" + "=" * 50)
print("CORRECTED TARGETED PGD RESULTS")
print("=" * 50)

print(f"Runtime:                {elapsed_time:.2f} seconds")
print(f"Initial target loss:    {loss_history[0]:.4f}")
print(f"Final target loss:      {final_loss:.4f}")
print(f"Best target loss:       {best_loss:.4f}")
print(f"Best iteration:         {best_iteration}")

print(f"\nConfigured epsilon:     {epsilon:.6f}")
print(f"Maximum perturbation:   {max_perturbation:.6f}")
print(f"Within constraint:      {within_constraint}")

print("\nOriginal transcription:   ", baseline_text)
print("Target transcription:     ", target_text)
print("Best transcription:       ", best_transcription)
print("Exact target match:       ", best_transcription == target_text)


Raw audio minimum: -0.375152587890625
Raw audio maximum: 0.285858154296875
Iteration  50/500 | Target CTC loss: 1.4715
Iteration 100/500 | Target CTC loss: 0.5673
Iteration 150/500 | Target CTC loss: 0.1406
Iteration 200/500 | Target CTC loss: 0.0511
Iteration 250/500 | Target CTC loss: 0.1104
Iteration 300/500 | Target CTC loss: 0.1240
Iteration 350/500 | Target CTC loss: 0.1018
Iteration 400/500 | Target CTC loss: 0.3268
Iteration 450/500 | Target CTC loss: 0.0789
Iteration 500/500 | Target CTC loss: 0.0011

CORRECTED TARGETED PGD RESULTS
Runtime:                14.04 seconds
Initial target loss:    26.9729
Final target loss:      0.0255
Best target loss:       0.0011
Best iteration:         500

Configured epsilon:     0.010000
Maximum perturbation:   0.010000
Within constraint:      True

Original transcription:    A STITCH IN TIME SAVES NINE BUT ACTION SPEAK LOUDER THAN WORDS
Target transcription:      OPEN THE SETTINGS MENU
Best transcription:        OPEN THE SETTINGS MENU
Exact 

## Step 9. Evaluate the Adversarial Transcription

In [ ]:
# Evaluate waveform after final update
with torch.no_grad():
    final_loss = target_ctc_loss(adversarial_audio).item()

if final_loss < best_loss:
    best_loss = final_loss
    best_audio = adversarial_audio.detach().clone()
    best_iteration = num_iterations + 1

elapsed_time = time.perf_counter() - start_time

# Decode best waveform
with torch.no_grad():
    logits = model(normalize_audio(best_audio)).logits
    predicted_ids = torch.argmax(logits, dim=-1)

best_transcription = processor.batch_decode(
    predicted_ids
)[0]

# Validate perturbation
delta = best_audio - original_audio
max_perturbation = delta.abs().max().item()

print("\n" + "=" * 50)
print("CORRECTED TARGETED PGD RESULTS")
print("=" * 50)

print(f"Runtime:                {elapsed_time:.2f} seconds")
print(f"Initial target loss:    {loss_history[0]:.4f}")
print(f"Final target loss:      {final_loss:.4f}")
print(f"Best target loss:       {best_loss:.4f}")
print(f"Best iteration:         {best_iteration}")

print(f"\nConfigured epsilon:     {epsilon:.6f}")
print(f"Maximum perturbation:   {max_perturbation:.6f}")
print(f"Within constraint:      {max_perturbation <= epsilon + 1e-6}")

print("\nOriginal transcription:   ", baseline_text)
print("Target transcription:     ", target_text)
print("Best transcription:       ", best_transcription)
print("Exact target match:       ", best_transcription == target_text)


CORRECTED TARGETED PGD RESULTS
Runtime:                11535.48 seconds
Initial target loss:    26.9729
Final target loss:      0.0255
Best target loss:       0.0011
Best iteration:         500

Configured epsilon:     0.010000
Maximum perturbation:   0.010000
Within constraint:      True

Original transcription:    A STITCH IN TIME SAVES NINE BUT ACTION SPEAK LOUDER THAN WORDS
Target transcription:      OPEN THE SETTINGS MENU
Best transcription:        OPEN THE SETTINGS MENU
Exact target match:        True


In [ ]:
# Create raw waveform tensor
original_audio = torch.tensor(
    waveform,
    dtype=torch.float32,
    device=device
).unsqueeze(0)

# Confirm raw waveform properties
print("Shape:", tuple(original_audio.shape))
print("Minimum:", original_audio.min().item())
print("Maximum:", original_audio.max().item())

# Verify transcription using the model's normal input processing
with torch.no_grad():
    normalized_audio = (
        original_audio - original_audio.mean(dim=-1, keepdim=True)
    ) / torch.sqrt(
        original_audio.var(dim=-1, keepdim=True, unbiased=False)
        + 1e-7
    )

    logits = model(normalized_audio).logits
    predicted_ids = torch.argmax(logits, dim=-1)

corrected_baseline = processor.batch_decode(predicted_ids)[0]

print("\nOriginal baseline: ", baseline_text)
print("Corrected baseline:", corrected_baseline)
print("Transcriptions match:", baseline_text == corrected_baseline)

Shape: (1, 65525)
Minimum: -0.375152587890625
Maximum: 0.285858154296875

Original baseline:  A STITCH IN TIME SAVES NINE BUT ACTION SPEAK LOUDER THAN WORDS
Corrected baseline: A STITCH IN TIME SAVES NINE BUT ACTION SPEAK LOUDER THAN WORDS
Transcriptions match: True


In [ ]:

from IPython.display import Audio, display

# Fixed playback gain applied equally to both recordings
playback_gain = 1

# Convert waveforms to NumPy arrays
original_samples = original_audio.detach().cpu().squeeze().numpy()
adversarial_samples = best_audio.detach().cpu().squeeze().numpy()

# Display waveform amplitude information
print("Original peak amplitude:", abs(original_samples).max())
print("Adversarial peak amplitude:", abs(adversarial_samples).max())

print("\nOriginal recording:")
display(Audio(
    original_samples * playback_gain,
    rate=sample_rate,
    normalize=False
))

print("\nAdversarial recording:")
display(Audio(
    adversarial_samples * playback_gain,
    rate=sample_rate,
    normalize=False
))


Original peak amplitude: 0.3751526
Adversarial peak amplitude: 0.37265274

Original recording:



Adversarial recording:
